In [1]:
# Imports
import torch
import torch.nn as nn
import torchvision

In [ ]:
# Let's load the dataset
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

#get the dataset (a bunch of flower images)
size = (64, 64)
transform = torchvision.transforms.Compose([torchvision.transforms.Resize(size), torchvision.transforms.ToTensor()])
train_dataset = list(torchvision.datasets.Flowers102("/tmp/flowers", "train", transform=transform, download=True))
test_dataset = list(torchvision.datasets.Flowers102("/tmp/flowers", "test", transform=transform, download=True))

#Merge the images into one big single image tensor and the labels into one big single label tensor
#Move to the device (GPU if available, otherwise CPU)
train_images = torch.stack([img for img, _ in train_dataset], dim=0).to(device)
test_images = torch.stack([img for img, _ in test_dataset], dim=0).to(device)
train_labels = torch.tensor([label for _, label in train_dataset]).to(device)
test_labels = torch.tensor([label for _, label in test_dataset]).to(device)

# Let's make sure we only have two classes
train_images, train_labels = train_images[train_labels < 2], train_labels[train_labels < 2]
test_images, test_labels = test_images[test_labels < 2], test_labels[test_labels < 2]

100.0%
100.0%
100.0%


In [ ]:
model2 = nn.Linear(64 * 64 * 3, 1).to(device)
loss2 = torch.nn.MSELoss()
optimizer2 = torch.optim.SGD(model2.parameters(), lr=0.01) #lr is the learning rate, which controls how much the model's parameters are updated during training
# Let's train the model
for epoch in range(10):
    #Compute model predictions
    pred_label = model2(train_images.view(train_images.size(0), -1))

    #Compute the loss
    loss_val = loss2(pred_label.view(-1), train_labels.float())
    
    #compute gradients
    optimizer2.zero_grad()
    loss_val.backward()
    optimizer2.step()
    print(f"Epoch {epoch}, Loss: {loss_val.item()}")

    #model2.train()
    #optimizer2.zero_grad()
    #outputs = model2(train_images.view(train_images.size(0), -1))
    #loss = loss2(outputs.squeeze(), train_labels.float())
    #loss.backward()
   # optimizer2.step()



   ####results is that it approaches infinite
   #because the the gradient give significant errors if the image's label is a 3 and the model predicts a 0
   #need to update this

Epoch 0, Loss: 1.19411301612854
Epoch 1, Loss: 1454.381103515625
Epoch 2, Loss: 2383986.0
Epoch 3, Loss: 3909349376.0
Epoch 4, Loss: 6410703208448.0
Epoch 5, Loss: 1.0512521941221376e+16
Epoch 6, Loss: 1.7238840390644138e+19
Epoch 7, Loss: 2.8268919220619136e+22
Epoch 8, Loss: 4.635648338979137e+25
Epoch 9, Loss: 7.601717053477119e+28


In [ ]:
#change the dataset to just have 0 and 1 as the labels to make gradient better

####This is what we change the dataset to just have 0 and 1 as the labels to make gradient better
train_images_01 = train_images[train_labels <= 1]
train_labels_01 = train_labels[train_labels <= 1]

model2 = nn.Linear(64 * 64 * 3, 1).to(device)
loss2 = torch.nn.MSELoss()
optimizer2 = torch.optim.SGD(model2.parameters(), lr=0.01) #lr is the learning rate, which controls how much the model's parameters are updated during training
# Let's train the model
for epoch in range(10):
    #Compute model predictions
    pred_label = model2(train_images_01.view(train_images_01.size(0), -1))

    #Compute the loss
    loss_val = loss2(pred_label.view(-1), train_labels_01.float())
    
    #compute gradients
    optimizer2.zero_grad()
    loss_val.backward()
    optimizer2.step()
    print(f"Epoch {epoch}, Loss: {loss_val.item()}")

    #model2.train()
    #optimizer2.zero_grad()
    #outputs = model2(train_images.view(train_images.size(0), -1))
    #loss = loss2(outputs.squeeze(), train_labels.float())
    #loss.backward()
   # optimizer2.step()


   #####result went to infinite again but slower
   #need to update the learning rate to a smaller number

Epoch 0, Loss: 0.8367297053337097
Epoch 1, Loss: 877.1222534179688
Epoch 2, Loss: 1437523.25
Epoch 3, Loss: 2357304320.0
Epoch 4, Loss: 3865599279104.0
Epoch 5, Loss: 6338958875164672.0
Epoch 6, Loss: 1.0394870889924526e+19
Epoch 7, Loss: 1.7045915172214655e+22
Epoch 8, Loss: 2.7952556514919904e+25
Epoch 9, Loss: 4.583768332253959e+28


In [ ]:
#change the the learning rate to a smaller number
####This is what we change the dataset to just have 0 and 1 as the labels to make gradient better
train_images_01 = train_images[train_labels <= 1]
train_labels_01 = train_labels[train_labels <= 1]

model2 = nn.Linear(64 * 64 * 3, 1).to(device)
loss2 = torch.nn.MSELoss()
#########this is where we change the learning rate to a smaller number
optimizer2 = torch.optim.SGD(model2.parameters(), lr=0.0001, momentum=0) #lr is the learning rate, which controls how much the model's parameters are updated during training
# Let's train the model
for epoch in range(10):
    #Compute model predictions
    pred_label = model2(train_images_01.view(train_images_01.size(0), -1))

    #Compute the loss
    loss_val = loss2(pred_label.view(-1), train_labels_01.float())
    
    #compute gradients
    optimizer2.zero_grad()
    loss_val.backward()
    optimizer2.step()
    print(f"Epoch {epoch}, Loss: {loss_val.item()}")

    #model2.train()
    #optimizer2.zero_grad()
    #outputs = model2(train_images.view(train_images.size(0), -1))
    #loss = loss2(outputs.squeeze(), train_labels.float())
    #loss.backward()
   # optimizer2.step()


   #####result much better as it slowly gets better and better results
   #can slowly increase the learning rate to find the optimal learning rate for this problem So try .0002 then .0003 and so on until it gives results that are increasing.
   #additionally you can traing longer by changing the number of epochs to get better results from 10


Epoch 0, Loss: 0.4435717463493347
Epoch 1, Loss: 0.36481237411499023
Epoch 2, Loss: 0.32937052845954895
Epoch 3, Loss: 0.3091926574707031
Epoch 4, Loss: 0.29465341567993164
Epoch 5, Loss: 0.28243574500083923
Epoch 6, Loss: 0.27138346433639526
Epoch 7, Loss: 0.26108068227767944
Epoch 8, Loss: 0.25136658549308777
Epoch 9, Loss: 0.24216827750205994


In [ ]:

test_images = torch.stack([img for img, _ in test_dataset], dim=0).to(device)
test_labels = torch.tensor([label for _, label in test_dataset]).to(device)

In [9]:
#use it on the test data
test_images_01 = test_images[test_labels <= 1]
test_labels_01 = test_labels[test_labels <= 1]


pred_label = model2(test_images_01.view(-1, 64 * 64 * 3))
print(loss2(pred_label.view(-1), test_labels_01.float()).item())





0.2961701452732086


In [3]:
def accuracy(pred: torch.Tensor, label: torch.Tensor) -> float:
    return ((pred > 0.5) == label).float().mean().item()


model = torch.nn.Linear(in_features=size[0] * size[1] * 3, out_features=1)
model = model.to(device)

loss_fn = nn.MSELoss()
optim = torch.optim.SGD(model.parameters(), lr=1e-4)
num_epochs = 500

for epoch in range(num_epochs):
    pred = model(train_images.view(train_images.shape[0], -1))[..., 0]
    loss_val = loss_fn(pred, train_labels.float())

    optim.zero_grad()
    loss_val.backward()
    optim.step()

    if epoch % 25 == 0 or epoch == num_epochs - 1:
        print(f"{epoch =:5d}  loss = {loss_val.item():.2f}  accuracy(train) = {accuracy(pred, train_labels):.3f}")

    if epoch % 100 == 0 or epoch == num_epochs - 1:
        with torch.inference_mode():
            pred = model(test_images.view(test_images.shape[0], -1))[..., 0]
            print(f"   Accuracy (test): {accuracy(pred, test_labels):.3f}")

epoch =    0  loss = 0.28  accuracy(train) = 0.500
   Accuracy (test): 0.350
epoch =   25  loss = 0.12  accuracy(train) = 0.900
epoch =   50  loss = 0.08  accuracy(train) = 0.900
epoch =   75  loss = 0.06  accuracy(train) = 0.900
epoch =  100  loss = 0.05  accuracy(train) = 0.950
   Accuracy (test): 0.717
epoch =  125  loss = 0.05  accuracy(train) = 0.950
epoch =  150  loss = 0.04  accuracy(train) = 0.950
epoch =  175  loss = 0.03  accuracy(train) = 0.950
epoch =  200  loss = 0.03  accuracy(train) = 0.950
   Accuracy (test): 0.733
epoch =  225  loss = 0.03  accuracy(train) = 0.950
epoch =  250  loss = 0.02  accuracy(train) = 1.000
epoch =  275  loss = 0.02  accuracy(train) = 1.000
epoch =  300  loss = 0.02  accuracy(train) = 1.000
   Accuracy (test): 0.733
epoch =  325  loss = 0.02  accuracy(train) = 1.000
epoch =  350  loss = 0.02  accuracy(train) = 1.000
epoch =  375  loss = 0.02  accuracy(train) = 1.000
epoch =  400  loss = 0.01  accuracy(train) = 1.000
   Accuracy (test): 0.733
epo